In [1]:
import pandas as pd
import numpy as np
import random
from Bio import SeqIO
from hmmlearn import hmm
import logging

import copy
import time

# Initialize PHMM

In [2]:
def init_transition_mat(match_len):
    order = ['M', 'D', 'I']
    COLS = [order[j] + f'{i+1}' for i in range(match_len - 1) for j in range (3)]
    COLS.insert(0, 'M0')
    COLS.insert(1, 'I0')
    COLS.append(f'M{match_len}')
    
    # INITIALIZE THE PROBABILITIES
    trans_matdf = pd.DataFrame(0, columns=COLS, index=COLS, dtype=np.float64)
    
    # Starting values: my own assumptions, not from a reference. Tune them.
    T_MM, T_MI, T_MD = 0.90, 0.05, 0.05      # from an internal Match state
    T_IM, T_II       = 0.50, 0.50            # from an internal Insert state
    T_DM, T_DD       = 0.90, 0.10            # from a Delete state
    BEGIN_M, BEGIN_D, BEGIN_I = 0.45, 0.05, 0.50   # from Begin (M0)
    FLANK_II         = 0.98                  # long self-loop for I0 and the last Insert (flanks)
     
    
    # Convention: trans_matdf.loc[TO, FROM]; each COLUMN sums to 1
    for j in range(match_len):
        if j == 0:                                        # Begin (M0) and I0
            trans_matdf.loc['M1', 'M0'] = BEGIN_M
            trans_matdf.loc['D1', 'M0'] = BEGIN_D
            trans_matdf.loc['I0', 'M0'] = BEGIN_I
            trans_matdf.loc['I0', 'I0'] = FLANK_II
            trans_matdf.loc['M1', 'I0'] = 1 - FLANK_II
            continue
    
        # from Match state j
        trans_matdf.loc[f'M{j+1}', f'M{j}'] = T_MM
        trans_matdf.loc[f'I{j}',   f'M{j}'] = T_MI
        if j + 1 <= match_len - 1:                                # D{n} doesn't exist
            trans_matdf.loc[f'D{j+1}', f'M{j}'] = T_MD
    
        # from Insert state j (the last one absorbs the C-terminal flank)
        if j == match_len - 1:
            trans_matdf.loc[f'I{j}',   f'I{j}'] = FLANK_II
            trans_matdf.loc[f'M{j+1}', f'I{j}'] = 1 - FLANK_II
        else:
            trans_matdf.loc[f'I{j}',   f'I{j}'] = T_II
            trans_matdf.loc[f'M{j+1}', f'I{j}'] = T_IM
    
        # from Delete state j
        trans_matdf.loc[f'M{j+1}', f'D{j}'] = T_DM
        if j + 1 <= match_len - 1:
            trans_matdf.loc[f'D{j+1}', f'D{j}'] = T_DD
    
    # loop back to the end state
    trans_matdf.loc[f'M{match_len}', f'M{match_len}'] = 1
    
    # After the loop: renormalize COLUMNS, because states near the end lose some targets
    col_sums = trans_matdf.sum(axis=0)
    trans_matdf = trans_matdf.div(col_sums.where(col_sums > 0, 1), axis=1)
    return trans_matdf

In [3]:
def init_emission_mat(match_len, conc=100.0, seed=None):
    rng = np.random.default_rng(seed)
    order = ['M', 'I']
    COLS = [order[j] + f'{i+1}' for i in range(match_len - 1) for j in range(2)]
    COLS.insert(0, 'M0')
    COLS.insert(1, 'I0')
    COLS.append(f'M{match_len}')

    amino = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
    symbols = ['^'] + amino + ['$']
    emission_matdf = pd.DataFrame(0.0, columns=COLS, index=symbols, dtype=np.float64)

    emission_matdf.loc[amino, :] = 1/20              # uniform over the 20 amino acids; '$' row stays 0

    # randomization: perturb each Match state around the uniform 1/20 (columns still sum to 1)
    for k in range(1, match_len):
        emission_matdf.loc[amino, f'M{k}'] = rng.dirichlet(np.full(20, conc / 20))

    emission_matdf['M0'] = 0.0                        # Begin: silent
    emission_matdf.loc['^', 'M0'] = 1.0
    emission_matdf[f'M{match_len}'] = 0.0             # End state: silent
    emission_matdf.loc['$', f'M{match_len}'] = 1.0    # emits only '$'

    return emission_matdf

In [5]:
def remove_silent_states(A, state_labels, silent_labels):
    silent_idx = [state_labels.index(s) for s in silent_labels]
    keep_idx = [i for i in range(len(state_labels)) if i not in silent_idx]
    kept_labels = [state_labels[i] for i in keep_idx]

    # slice the four corners
    A_EE = A[np.ix_(keep_idx, keep_idx)]      # real   -> real
    A_ED = A[np.ix_(keep_idx, silent_idx)]    # silent -> real   (exit)
    A_DE = A[np.ix_(silent_idx, keep_idx)]    # real   -> silent (entry)
    A_DD = A[np.ix_(silent_idx, silent_idx)]  # silent -> silent

    # fold all detours through the silent states back into A_EE
    n_silent = len(silent_idx)
    fundamental = np.linalg.inv(np.eye(n_silent) - A_DD)   # (I - A_DD)^-1
    correction = A_ED @ fundamental @ A_DE

    A_reduced = A_EE + correction
    return A_reduced, kept_labels

In [6]:
family_df = pd.read_csv('protein_families.csv')
family_df

,name,acc,leng,nseq,type,clan
0,AraC_N,PF06719,150,195,Family,CL0029
1,CAP_C,PF08603,157,131,Family,CL0391
2,DUF5683,PF18935,151,287,Family,CL0934
3,DUF6580,PF20221,162,214,Family,CL0315
4,HisG,PF01634,158,509,Family,CL0177
5,MksE,PF21980,163,210,Family,CL0123
6,OxoGdeHyase_C,PF16870,151,211,Family,CL0591
7,PHP,PF02811,164,433,Family,CL0036
8,Reticulon,PF02453,157,194,Family,CL0484
9,RraA-like,PF03737,151,278,Family,CL0364


In [7]:
i = 0

name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]
trans_matdf = init_transition_mat(match_len)
cols = list(trans_matdf.columns)
d_cols = [i for i in cols if i[0] == 'D']
trans_matrix, red_cols = remove_silent_states(np.array(trans_matdf), cols, d_cols)
trans_matdf = pd.DataFrame(trans_matrix, columns=red_cols, index=red_cols)
emit_matdf = init_emission_mat(match_len)

In [8]:
trans_matdf.to_csv('trans_mat.csv')
emit_matdf.to_csv('emit_mat.csv')

# Build the Profile HMM

In [9]:
# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

In [10]:
# --- hmmlearn wants row-stochastic matrices; all 301 states are kept ---------------------
A = trans_matdf.to_numpy().T                 # A[from, to]
n = A.shape[0]
 
# hmmlearn has no silent states: every state emits at every step. So M0 emits a dummy
# begin symbol '^' (mirroring '$' for M150), and '^' is prepended to every sequence.
BEGIN = sym_idx['^']
END = sym_idx['$']
B = emit_matdf.to_numpy().T                  # B[state, symbol]
assert B[0, BEGIN] == 1.0 and np.count_nonzero(B[:, BEGIN]) == 1, "'^' must be emitted only by M0"
assert B[-1, END] == 1.0 and np.count_nonzero(B[:, END]) == 1, "'$' must be emitted only by M150"

startprob = np.zeros(n)
startprob[0] = 1.0                           # every path starts in M0

In [11]:
def build_model(trans_strength=5.0, emit_strength=20.0, A0=A, B0=B):
    """Profile HMM initialised at (A0, B0), with Dirichlet priors centred on them.
 
    prior = 1 + strength * initial_probs. Entries that start at 0 get no pseudocount, so
    '$' and '^' never leak out of M150 and M0, and the topology is untouched; M150 -> M150
    gets a pseudocount, which keeps its row valid even though no transition out of M150 is
    ever observed. The priors only add pseudocounts in the M-step; they don't change the
    starting values.
    """
    m = hmm.CategoricalHMM(
        n_components=n,
        n_features=len(symbols),
        startprob_prior=1.0 + trans_strength * startprob,
        transmat_prior=1.0 + trans_strength * A0,
        emissionprob_prior=1.0 + emit_strength * B0,
        init_params="",            # default "ste" would overwrite the matrices below with random ones
        params="ste",              # what Baum-Welch updates
        n_iter=1,                  # one EM step per fit() call; train() handles convergence
        implementation="scaling",  # several times faster than "log" here, same result
    )
    m.startprob_ = startprob.copy()
    m.transmat_ = A0.copy()
    m.emissionprob_ = B0.copy()
    return m

In [12]:
model = build_model()

# Read Fasta File

In [13]:
# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse("sequences/PF06719.fasta", "fasta")]

# Split into Training and Testing Set

In [14]:
def split_train_val_test(records, train_fraction=0.7, val_fraction=0.1, seed=42):
    """records: list of (header, sequence).
    Returns (train_seqs, val_seqs, test_seqs) as lists of strings.
    Test gets the remainder: 1 - train_fraction - val_fraction."""
    if train_fraction < 0 or val_fraction < 0 or train_fraction + val_fraction > 1 + 1e-9:
        raise ValueError("fractions must be non-negative and sum to at most 1")
    records = list(records)                      # copy, so the original list is not reordered
    random.Random(seed).shuffle(records)
    n = len(records)
    n_train = round(train_fraction * n)
    n_train_val = round((train_fraction + val_fraction) * n)   # cumulative cut point
    train = [s for _, s in records[:n_train]]
    val   = [s for _, s in records[n_train:n_train_val]]
    test  = [s for _, s in records[n_train_val:]]
    return train, val, test

In [15]:
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

# Encode Sequences

In [16]:
def encode(seqs):
    """Strings -> hmmlearn input. Wraps each sequence as '^' + seq + '$', so every path
    has to start in M0 and end in M150."""
    encoded = [[BEGIN] + [sym_idx[c] for c in s.upper()] + [END] for s in seqs]
    return np.concatenate(encoded).reshape(-1, 1), [len(e) for e in encoded]

In [20]:
BEGIN

0

# Train the PHMM

In [1]:
def log_prior(m):
    """Log Dirichlet prior density of the current parameters (up to a constant)."""
    lp = 0.0
    for prior, p in [(m.startprob_prior, m.startprob_), (m.transmat_prior, m.transmat_),
                     (m.emissionprob_prior, m.emissionprob_)]:
        mask = prior > 1
        lp += np.sum((prior[mask] - 1) * np.log(p[mask]))
    return lp
 
 
# --- Description accuracy (Zhao & Jaeger 2007; Adhikary et al. 2020, Sec. 4) ------------
# DA = f(1 + log_s P(seq) / l),  f(x) = tanh(x/8) for x <= 0,  f(x) = x for x > 0.
# s and l count only the symbols the model actually predicts: the 20 amino acids plus '$'
# ('^' is given, and contributes log 1 = 0). So a model that guesses uniformly among those
# 21 symbols scores DA = 0, DA > 0 is better than that, and DA = 1 is perfect prediction.
S_PREDICTED = len(symbols) - 1
 
 
def description_accuracy(m, seqs):
    """Per-sequence DA; the paper reports the mean and the std across sequences."""
    da = np.empty(len(seqs))
    for i, s in enumerate(seqs):
        X, _ = encode([s])
        print(X)
        try:
            ll = m.score(X)                  # natural-log P('^' + s + '$')
        except ValueError:                   # zero-probability sequence ('scaling' raises)
            ll = -np.inf
        x = 1.0 + ll / (np.log(S_PREDICTED) * (len(s) + 1))   # l = residues + '$'
        da[i] = x if x > 0 else np.tanh(x / 8.0)
    return da
 
 
def train(m, train_seqs, val_seqs=None, test_seqs=None, max_iter=200, tol=1e-3, verbose=True):
    """Baum-Welch (MAP), one EM step at a time.
 
    Convergence is tracked on log-likelihood + log-prior, which EM never decreases;
    hmmlearn's built-in monitor uses the likelihood alone, which can dip when priors are
    used and would stop training early.
 
    Returns (model, history). history has one row per iteration (row 0 = untrained model)
    with the CPU time spent in fit() so far, the training log-likelihood/log-posterior and,
    if given, the mean/std DA on val_seqs and test_seqs. As in the paper, when val_seqs is
    given the returned model has the parameters from the iteration with the best
    validation DA; test DA is only recorded, never used for selection.
    """
    X, lengths = encode(train_seqs)
    history, cpu = [], 0.0
 
    def record(it):
        ll = m.score(X, lengths)
        row = {"iter": it, "cpu_time": cpu, "loglik": ll, "logpost": ll + log_prior(m)}
        for name, seqs in (("val", val_seqs), ("test", test_seqs)):
            if seqs:
                da = description_accuracy(m, seqs)
                row[f"{name}_da_mean"], row[f"{name}_da_std"] = da.mean(), da.std()
        history.append(row)
        if verbose:
            extra = "".join(f"  {k} {v:.4f}" for k, v in row.items() if k.endswith("da_mean"))
            print(f"iter {it:3d}  cpu {cpu:7.1f}s  log-lik {ll:.2f}  log-posterior {row['logpost']:.2f}{extra}")
        return row
 
    def params():
        return copy.deepcopy((m.startprob_, m.transmat_, m.emissionprob_))
 
    best = (record(0).get("val_da_mean", -np.inf), params())
    for it in range(1, max_iter + 1):
        t0 = time.process_time()
        m.fit(X, lengths)
        cpu += time.process_time() - t0
        row = record(it)
        if val_seqs and row["val_da_mean"] > best[0]:
            best = (row["val_da_mean"], params())
        if row["logpost"] - history[-2]["logpost"] < tol:
            break
    if val_seqs:
        m.startprob_, m.transmat_, m.emissionprob_ = best[1]
    return m, pd.DataFrame(history)

NameError: name 'symbols' is not defined

In [20]:
logging.getLogger("hmmlearn").setLevel(logging.ERROR)  # silences the "free scalar parameters" warning,
model, history_df = train(model, train_seqs)

KeyboardInterrupt: 

In [18]:
history_df

,iter,cpu_time,loglik,logpost
0,0,0.000000,-64551.655533,-83011.635023
1,1,7.385699,-59243.686391,-78226.299163
2,2,14.867814,-53165.460478,-72832.045396
3,3,22.432077,-49562.477218,-70143.746776
4,4,30.264568,-48906.480162,-69699.734568
...,...,...,...,...
58,58,483.214854,-48051.112276,-69002.205308
59,59,490.931772,-48051.108278,-69002.204024
60,60,498.559591,-48051.104346,-69002.202874
61,61,506.334983,-48051.100500,-69002.201835


In [19]:
"""Evaluate profile-HMM training the way Adhikary et al. (2020) evaluate HQMMs:
description accuracy (DA) on held-out sequences, the model with the best validation DA,
and two figures in the style of their Fig. 1 (bar chart) and Fig. 2 (DA vs. CPU time)."""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# from train_profile_hmm import build_model, description_accuracy, train

try:
    plt.style.use("seaborn-v0_8-darkgrid")   # close to the paper's look
except OSError:
    pass


def run_experiment(settings, train_seqs, val_seqs, test_seqs, max_iter=200, tol=1e-3):
    """settings: {label: (trans_strength, emit_strength)}.

    Trains one model per setting and returns
      histories: {label: per-iteration DataFrame from train()}
      results:   {label: (mean test DA, std test DA)} for each setting's best-validation
                 model, plus the untrained starting model under "untrained".
    """
    da0 = description_accuracy(build_model(), test_seqs)
    results = {"untrained": (da0.mean(), da0.std())}
    histories = {}
    for label, (ts, es) in settings.items():
        print(f"--- {label}: trans_strength={ts}, emit_strength={es}")
        model, hist = train(build_model(ts, es), train_seqs, val_seqs, test_seqs, max_iter, tol)
        da = description_accuracy(model, test_seqs)
        results[label] = (da.mean(), da.std())
        histories[label] = hist
    return histories, results


def _references(ax, references):
    for i, (label, value) in enumerate((references or {}).items()):
        ax.axhline(value, ls=["--", ":", "-."][i % 3], color="0.25", lw=1.2, label=label, zorder=1)


def plot_da_vs_time(histories, references=None, x="cpu_time", out="da_vs_time.png"):
    """Fig. 2 style: test DA vs CPU training time (or x="iter"). The line is the mean over
    test sequences and the band is +/- 1 std across test sequences."""
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for label, h in histories.items():
        (line,) = ax.plot(h[x], h["test_da_mean"], marker="o", ms=3, lw=1.8, label=label)
        ax.fill_between(h[x], h["test_da_mean"] - h["test_da_std"],
                        h["test_da_mean"] + h["test_da_std"], color=line.get_color(), alpha=0.18)
    _references(ax, references)
    ax.set_xlabel("CPU time (s)" if x == "cpu_time" else "EM iteration")
    ax.set_ylabel("Test DA")
    ax.set_title("Test DA during Baum-Welch training")
    ax.legend(loc="lower right", frameon=True)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


def plot_da_bars(results, references=None, out="da_bars.png"):
    """Fig. 1 style: test DA of each model; error bars = 1 std of DA across test sequences.
    Like the paper, the y-axis is zoomed to the range of the results."""
    labels = list(results)
    means = np.array([results[k][0] for k in labels])
    stds = np.array([results[k][1] for k in labels])
    fig, ax = plt.subplots(figsize=(max(4.5, 1.3 * len(labels) + 1.5), 4.2))
    trained = [k for k in labels if k != "untrained"]           # same colours as plot_da_vs_time
    colors = ["0.65" if k == "untrained" else f"C{trained.index(k)}" for k in labels]
    ax.bar(range(len(labels)), means, yerr=stds, capsize=5, color=colors, width=0.6,
           error_kw={"elinewidth": 1.2, "ecolor": "0.2"}, zorder=2)
    _references(ax, references)
    lo = min((means - stds).min(), *(references or {}).values()) if references else (means - stds).min()
    hi = max((means + stds).max(), *(references or {}).values()) if references else (means + stds).max()
    ax.set_ylim(lo - 0.1 * (hi - lo), hi + 0.25 * (hi - lo))   # headroom for the legend
    ax.set_xticks(range(len(labels)), labels)
    ax.set_ylabel("Test DA")
    ax.set_title("Test DA by prior strength")
    if references:
        ax.legend(loc="upper left", frameon=True)
    fig.tight_layout()
    fig.savefig(out, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    # your sequences as strings, WITHOUT '^' or '$'
    # train_seqs = ["..."]
    # val_seqs = ["..."]
    # test_seqs = ["..."]
    logging.getLogger("hmmlearn").setLevel(logging.ERROR)  # silences the "free scalar parameters" warning,

    # what to compare: label -> (TRANS_STRENGTH, EMIT_STRENGTH)
    settings = {
        "weak prior (1, 5)": (1.0, 5.0),
        "default (5, 20)": (5.0, 20.0),
        "strong prior (20, 80)": (20.0, 80.0),
    }
    # horizontal dashed lines, e.g. {"true model": 0.31} if your data was simulated
    # from a known model (the paper's dashed line); leave empty otherwise
    references = {}

    histories, results = run_experiment(settings, train_seqs, val_seqs, test_seqs)
    for label, h in histories.items():
        h.to_csv(f"history_{label.split()[0]}.csv", index=False)
    pd.DataFrame(results, index=["test_da_mean", "test_da_std"]).T.to_csv("test_da_results.csv")
    plot_da_vs_time(histories, references, out="da_vs_time.png")
    plot_da_bars(results, references, out="da_bars.png")

--- weak prior (1, 5): trans_strength=1.0, emit_strength=5.0
iter   0  cpu     0.0s  log-lik -64514.15  log-posterior -69085.70  val_da_mean 0.0096  test_da_mean 0.0093
iter   1  cpu     8.0s  log-lik -59496.81  log-posterior -64283.14  val_da_mean 0.0912  test_da_mean 0.0894
iter   2  cpu    17.5s  log-lik -54353.92  log-posterior -59449.54  val_da_mean 0.1727  test_da_mean 0.1736
iter   3  cpu    24.6s  log-lik -50346.84  log-posterior -55815.24  val_da_mean 0.2350  test_da_mean 0.2325
iter   4  cpu    31.8s  log-lik -48198.94  log-posterior -53904.08  val_da_mean 0.2638  test_da_mean 0.2604
iter   5  cpu    39.4s  log-lik -47440.46  log-posterior -53223.30  val_da_mean 0.2725  test_da_mean 0.2699


KeyboardInterrupt: 